# System 1 Notebook 00 — Master ingestion & batch assignment

Notebook này là bước đầu tiên của System 1.

Nhiệm vụ chính:

1. Chuẩn bị runtime cho Kaggle / Colab / local.
2. Clone hoặc dùng repo `system1` hiện có.
3. Cấu hình input / output / artifact store.
4. Chạy phase00:

   * ingest dữ liệu raw video + metadata
   * tạo release skeleton
   * tạo media mapping
   * chia batch cho worker notebooks
5. Kiểm tra output và checkpoint status.

Notebook này **không xử lý structure/features/merge**. Các bước đó nằm ở Notebook 01/02/03.

Nguyên tắc:

* Người dùng chỉ chỉnh section config.
* Các cell phía dưới không nên sửa nếu không debug.
* Notebook chỉ orchestration; logic thật nằm trong package `system1`.


# Section 1 — User config: phần người dùng thường chỉnh

Chỉnh các biến trong cell này trước khi chạy notebook.

- `mock` dùng để kiểm tra pipeline không cần model thật.
- `AIC_NUM_BATCHES` là số batch chia cho worker.
- `GITHUB_BRANCH` nên là `system1` khi test pipeline hiện tại.
- Giữ `providers`, `worker_id`, `batch_id` để tương thích notebook/test và các notebook worker.

In [ ]:
from __future__ import annotations
import os

In [ ]:
# ============================================================
# Chỉ sửa các biến trong cell này.

# Các cell phía dưới nên giữ nguyên nếu không debug.

# 1. GitHub repo config

# Nếu repo public: giữ nguyên.

# Nếu repo private: set GITHUB_TOKEN trong Kaggle/Colab Secret hoặc local env.

GITHUB_REPO_URL = os.environ.get(

    "GITHUB_REPO_URL",

    "https://github.com/awun0105/Multimodal-Agentic-Retrieval-Engine.git",

)



# Branch đang chứa code System 1.

# Khi code đã merge main thì đổi thành "main".

GITHUB_BRANCH = os.environ.get("GITHUB_BRANCH", "system1")


# ------------------------------------------------------------

# 2. Input source config

# ------------------------------------------------------------

# Nếu đã attach Kaggle Dataset hoặc đã có sẵn raw_videos/metadata:

#   để trống AIC_ORGANIZER_SOURCE_URI.

#

# Nếu cần import từ Google Drive folder/public source:

#   set AIC_ORGANIZER_SOURCE_URI = "https://drive.google.com/drive/folders/..."

AIC_ORGANIZER_SOURCE_URI = os.environ.get("AIC_ORGANIZER_SOURCE_URI", "")





# ------------------------------------------------------------

# 3. Provider configuration

# ------------------------------------------------------------

# mock: không cần model thật, dùng để test pipeline

# real/config/rule_based/vlm: dùng cho các phase sau nếu đã cấu hình model

providers = os.environ.get("AIC_PROVIDERS", "mock")



# Số batch chia cho worker notebooks.

# Debug một worker thì để 1.

AIC_NUM_BATCHES = int(os.environ.get("AIC_NUM_BATCHES", "1"))





# ------------------------------------------------------------

# 4. Release / worker identity

# ------------------------------------------------------------

# Release id là tên folder release nằm trong output root.

AIC_RELEASE_ID = os.environ.get("AIC_RELEASE_ID", "competition_dataset_v001")



# Notebook 00 thường là master, nhưng vẫn giữ worker_id để thống nhất runtime.

AIC_WORKER_ID = os.environ.get("AIC_WORKER_ID", "worker_master_00")



# Notebook 00 tạo batch files; batch_id chủ yếu dùng cho handoff/debug.

AIC_BATCH_ID = os.environ.get("AIC_BATCH_ID", "batch_000")





# ------------------------------------------------------------

# 5. Optional path override

# ------------------------------------------------------------

# Để trống để notebook tự chọn theo môi trường:

#   Kaggle: /kaggle/working/input, /kaggle/working/output

#   Colab: /content/input, /content/output

#   Local: system1/input, system1/output

#

# Nếu muốn override, set path ở đây hoặc set env trước khi chạy notebook.

AIC_DATA_ROOT_OVERRIDE = os.environ.get("AIC_DATA_ROOT", "")

AIC_RUNTIME_ROOT_OVERRIDE = os.environ.get("AIC_RUNTIME_ROOT", "")

AIC_ARTIFACT_ROOT_OVERRIDE = os.environ.get("AIC_ARTIFACT_ROOT", "")





# ------------------------------------------------------------

# 6. Artifact / checkpoint backend

# ------------------------------------------------------------

# local: lưu checkpoint ở máy/session hiện tại.

# hf_dataset: lưu checkpoint lên Hugging Face Dataset repo.

AIC_ARTIFACT_BACKEND = os.environ.get("AIC_ARTIFACT_BACKEND", "local")



# Chỉ cần set khi AIC_ARTIFACT_BACKEND = "hf_dataset".

# Ví dụ: "awun0105/hcmai-system1-artifacts"

AIC_HF_REPO_ID = os.environ.get("AIC_HF_REPO_ID", "")



# Thường giữ nguyên.

AIC_HF_REPO_TYPE = os.environ.get("AIC_HF_REPO_TYPE", "dataset")

AIC_HF_REVISION = os.environ.get("AIC_HF_REVISION", "main")



# Prefix trong HF repo để tránh lẫn nhiều release.

# Khuyến nghị dùng AIC_RELEASE_ID.

AIC_HF_PREFIX = os.environ.get("AIC_HF_PREFIX", AIC_RELEASE_ID)



# ------------------------------------------------------------

# 7. Resume / sync behavior

# ------------------------------------------------------------

# AIC_RESUME=true:

#   nếu có checkpoint phase00 thì restore và skip phần đã chạy.

#

# AIC_SYNC=true:

#   sau assign-batches sẽ save checkpoint phase00.

#

# AIC_FORCE_REBUILD=false:

#   mặc định không ép rebuild nếu có checkpoint.

AIC_RESUME = os.environ.get("AIC_RESUME", "true")

AIC_SYNC = os.environ.get("AIC_SYNC", "true")

AIC_FORCE_REBUILD = os.environ.get("AIC_FORCE_REBUILD", "false")



# Section 2 — Detect môi trường và thiết lập workspace

Cell này tự nhận diện môi trường chạy.

- Kaggle dùng `/kaggle/working` làm workspace mặc định.
- Colab dùng `/content` làm workspace mặc định; nếu muốn output bền vững thì mount Google Drive.
- Local dùng thư mục hiện tại hoặc `AIC_WORKSPACE_ROOT`.

In [ ]:
from pathlib import Path


def detect_environment() -> str:
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    try:
        import google.colab  # type: ignore  # noqa: F401
        return "colab"
    except Exception:
        return "local"


ENVIRONMENT = detect_environment()
if ENVIRONMENT == "kaggle":
    default_workspace_root = Path("/kaggle/working")
elif ENVIRONMENT == "colab":
    default_workspace_root = Path("/content")
else:
    default_workspace_root = Path.cwd()

WORKSPACE_ROOT = Path(os.environ.get("AIC_WORKSPACE_ROOT", str(default_workspace_root))).expanduser().resolve()
AIC_REPO_PARENT = Path(os.environ.get("AIC_REPO_PARENT", str(WORKSPACE_ROOT))).expanduser().resolve()
AIC_REPO_PARENT.mkdir(parents=True, exist_ok=True)

print({
    "environment": ENVIRONMENT,
    "workspace_root": str(WORKSPACE_ROOT),
    "AIC_REPO_PARENT": str(AIC_REPO_PARENT),
})

# Section 3 — Secret và repo private

Nếu repo public thì không cần set secret.

Nếu repo private:

- Kaggle: Add-ons / Secrets → tạo `GITHUB_TOKEN`.
- Colab: Secrets → tạo `GITHUB_TOKEN`.
- Local: `export GITHUB_TOKEN=...`.

Không hardcode token vào notebook và không print token ra log.

In [ ]:
from urllib.parse import urlparse, urlunparse


def get_secret(name: str) -> str | None:
    value = os.environ.get(name)
    if value:
        return value
    if ENVIRONMENT == "kaggle":
        try:
            from kaggle_secrets import UserSecretsClient  # type: ignore
            return UserSecretsClient().get_secret(name)
        except Exception:
            return None
    if ENVIRONMENT == "colab":
        try:
            from google.colab import userdata  # type: ignore
            return userdata.get(name)
        except Exception:
            return None
    return None


def auth_repo_url(url: str) -> str:
    token = get_secret("GITHUB_TOKEN")
    parsed = urlparse(url)
    if token and parsed.scheme == "https" and parsed.netloc == "github.com":
        return urlunparse((parsed.scheme, f"x-access-token:{token}@{parsed.netloc}", parsed.path, "", "", ""))
    return url


def safe_repo_url(url: str) -> str:
    parsed = urlparse(url)
    if "@" in parsed.netloc:
        safe_netloc = "***@" + parsed.netloc.split("@", 1)[1]
        return urlunparse((parsed.scheme, safe_netloc, parsed.path, "", "", ""))
    return url


print("repo_url=", safe_repo_url(auth_repo_url(GITHUB_REPO_URL)))

# Section 4 — Clone/pull repo và checkout branch

Cell này clone repo nếu chưa có, hoặc fetch/checkout/pull đúng branch nếu repo đã tồn tại.

Lưu ý:

- Nếu đang sửa code trực tiếp trong runtime, `git pull --ff-only` có thể fail nếu có thay đổi local.
- Khi test branch hiện tại, giữ `GITHUB_BRANCH = "system1"`.
- Kaggle ưu tiên workspace `/kaggle/working`, không dùng `/content`.

In [ ]:
import shutil
import subprocess
import sys

REPO_DIR_NAME = Path(urlparse(GITHUB_REPO_URL).path).stem or "Multimodal-Agentic-Retrieval-Engine"


def run_shell(command: list[str], *, cwd: Path | None = None, safe_command: list[str] | None = None) -> None:
    shown = safe_command or command
    print("$", " ".join(str(part) for part in shown))
    subprocess.run(command, cwd=str(cwd) if cwd else None, check=True)


def find_system1_root(repo_root: Path) -> Path | None:
    if (repo_root / "system1" / "pyproject.toml").exists():
        return repo_root / "system1"
    if (repo_root / "pyproject.toml").exists() and (repo_root / "src" / "system1").exists():
        return repo_root
    return None


def current_tree_repo_root() -> Path | None:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if find_system1_root(candidate) is not None:
            return candidate
    return None


def resolve_repo_root() -> Path:
    env_repo_root = os.environ.get("AIC_REPO_ROOT")
    if env_repo_root:
        candidate = Path(env_repo_root).expanduser().resolve()
        if find_system1_root(candidate) is not None:
            return candidate
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "system1").exists():
            return candidate
        raise FileNotFoundError(f"AIC_REPO_ROOT không chứa package system1 hợp lệ: {candidate}")

    local_repo_root = current_tree_repo_root()
    if ENVIRONMENT == "local" and local_repo_root is not None:
        return local_repo_root

    return AIC_REPO_PARENT / REPO_DIR_NAME


REPO_ROOT = resolve_repo_root()
SYSTEM1_ROOT = find_system1_root(REPO_ROOT)
clone_url = auth_repo_url(GITHUB_REPO_URL)
safe_clone_url = safe_repo_url(clone_url)

if SYSTEM1_ROOT is not None and not (REPO_ROOT / ".git").exists():
    print(f"Dùng repo/package local có sẵn: {REPO_ROOT}")
elif REPO_ROOT.exists() and (REPO_ROOT / ".git").exists():
    # Nếu repo đã tồn tại và là git repo, cập nhật đúng branch thay vì clone lại.
    run_shell(["git", "fetch", "origin"], cwd=REPO_ROOT)
    run_shell(["git", "checkout", GITHUB_BRANCH], cwd=REPO_ROOT)
    run_shell(["git", "pull", "--ff-only", "origin", GITHUB_BRANCH], cwd=REPO_ROOT)
elif REPO_ROOT.exists() and find_system1_root(REPO_ROOT) is None:
    # Nếu folder clone bị lỗi giữa chừng, xóa để tránh dùng trạng thái hỏng.
    shutil.rmtree(REPO_ROOT)
    run_shell(
        ["git", "clone", "--branch", GITHUB_BRANCH, clone_url, str(REPO_ROOT)],
        safe_command=["git", "clone", "--branch", GITHUB_BRANCH, safe_clone_url, str(REPO_ROOT)],
    )
elif not REPO_ROOT.exists():
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    run_shell(
        ["git", "clone", "--branch", GITHUB_BRANCH, clone_url, str(REPO_ROOT)],
        safe_command=["git", "clone", "--branch", GITHUB_BRANCH, safe_clone_url, str(REPO_ROOT)],
    )

SYSTEM1_ROOT = find_system1_root(REPO_ROOT)
if SYSTEM1_ROOT is None:
    raise FileNotFoundError(f"Không tìm thấy package system1 trong repo: {REPO_ROOT}")

print({"repo_root": str(REPO_ROOT), "system1_root": str(SYSTEM1_ROOT), "github_branch": GITHUB_BRANCH})

# Section 5 — Cài package System 1

Cell này cài package bằng editable install.

- `-e` nghĩa là editable install: notebook gọi CLI `system1`, nhưng code chạy từ package vừa cài.
- `SYSTEM1_ROOT` phải là package root có `pyproject.toml`.

In [ ]:
if not (SYSTEM1_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError(f"SYSTEM1_ROOT không có pyproject.toml: {SYSTEM1_ROOT}")

run_shell([sys.executable, "-m", "pip", "install", "-q", "gdown", "-e", str(SYSTEM1_ROOT)])

os.environ["AIC_REPO_ROOT"] = str(REPO_ROOT)
os.environ["AIC_SYSTEM1_ROOT"] = str(SYSTEM1_ROOT)

# Section 6 — Thiết lập data/output/runtime paths

Cell này tạo các biến đường dẫn runtime.

- Kaggle mặc định dùng `AIC_DATA_ROOT = /kaggle/working/input`, `AIC_RUNTIME_ROOT = /kaggle/working/output`.
- Colab mặc định dùng `AIC_DATA_ROOT = /content/input`, `AIC_RUNTIME_ROOT = /content/output`.
- Local mặc định dùng `SYSTEM1_ROOT / "input"` và `SYSTEM1_ROOT / "output"`.

Ví dụ:

Kaggle Dataset attach:

```python
os.environ["AIC_DATA_ROOT"] = "/kaggle/input/<dataset-name>"
os.environ["AIC_RUNTIME_ROOT"] = "/kaggle/working/output"
```

Kaggle Google Drive import:

```python
os.environ["AIC_ORGANIZER_SOURCE_URI"] = "https://drive.google.com/drive/folders/..."
os.environ["AIC_DATA_ROOT"] = "/kaggle/working/input"
os.environ["AIC_RUNTIME_ROOT"] = "/kaggle/working/output"
```

Colab Drive:

```python
from google.colab import drive
drive.mount("/content/drive")
os.environ["AIC_DATA_ROOT"] = "/content/drive/MyDrive/aic/system1/input"
os.environ["AIC_RUNTIME_ROOT"] = "/content/drive/MyDrive/aic/system1/output"
```

Local:

```python
os.environ["AIC_REPO_ROOT"] = "/path/to/Multimodal-Agentic-Retrieval-Engine"
os.environ["AIC_DATA_ROOT"] = "/path/to/input"
os.environ["AIC_RUNTIME_ROOT"] = "/path/to/output"
```

In [ ]:
if ENVIRONMENT == "kaggle":
    default_data_root = Path("/kaggle/working/input")
    default_runtime_root = Path("/kaggle/working/output")
elif ENVIRONMENT == "colab":
    default_data_root = Path("/content/input")
    default_runtime_root = Path("/content/output")
else:
    default_data_root = SYSTEM1_ROOT / "input"
    default_runtime_root = SYSTEM1_ROOT / "output"

AIC_DATA_ROOT = Path(os.environ.get("AIC_DATA_ROOT", str(default_data_root))).expanduser().resolve()
AIC_RUNTIME_ROOT = Path(os.environ.get("AIC_RUNTIME_ROOT", str(default_runtime_root))).expanduser().resolve()
AIC_ARTIFACT_ROOT = Path(os.environ.get("AIC_ARTIFACT_ROOT", str(AIC_RUNTIME_ROOT))).expanduser().resolve()

input_dir = AIC_DATA_ROOT
output_dir = AIC_RUNTIME_ROOT
release_dir = output_dir / "competition_dataset_v001"
RELEASE_DIR = release_dir

AIC_DATA_ROOT.mkdir(parents=True, exist_ok=True)
AIC_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
AIC_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

os.environ["AIC_DATA_ROOT"] = str(AIC_DATA_ROOT)
os.environ["AIC_RUNTIME_ROOT"] = str(AIC_RUNTIME_ROOT)
os.environ["AIC_ARTIFACT_ROOT"] = str(AIC_ARTIFACT_ROOT)

print({"input_dir": str(input_dir), "output_dir": str(output_dir), "release_dir": str(release_dir)})

# Section 7 — Helper: chạy CLI và đọc JSON

Cell này định nghĩa helper dùng chung.

- `run_cli` luôn in command trước khi chạy.
- Command chạy ở `cwd=SYSTEM1_ROOT`.
- `check=True` giúp fail fast nếu CLI lỗi.

In [ ]:
import json
from typing import Any


def system1_executable() -> list[str]:
    binary = shutil.which("system1")
    if binary:
        return [binary]
    return [sys.executable, "-m", "system1.cli"]


def run_cli(args: list[str]) -> None:
    command = [*system1_executable(), *args]
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=str(SYSTEM1_ROOT), check=True)


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def path_summary() -> None:
    print({
        "environment": ENVIRONMENT,
        "repo_root": str(REPO_ROOT),
        "system1_root": str(SYSTEM1_ROOT),
        "github_branch": GITHUB_BRANCH,
        "input_dir": str(input_dir),
        "output_dir": str(output_dir),
        "release_dir": str(release_dir),
        "providers": providers,
        "num_batches": num_batches,
        "worker_id": worker_id,
        "batch_id": batch_id,
    })


path_summary()

# Section 8 — Kiểm tra hoặc import input dataset

Cell này kiểm tra input layout trước khi chạy ingest.

Tùy môi trường:

- Kaggle Dataset attach: set `AIC_DATA_ROOT = "/kaggle/input/<dataset-name>"`; dataset phải có trực tiếp `raw_videos/` và `metadata/`.
- Google Drive public folder: set `AIC_ORGANIZER_SOURCE_URI`.
- Local: đặt input theo layout chuẩn hoặc set path tới folder đã có.

Lưu ý quan trọng: `import-source` có thể reset `raw_videos/` và `metadata/` trong `AIC_DATA_ROOT`. Không đặt `AIC_DATA_ROOT` vào thư mục chứa dữ liệu khác cần giữ.

In [ ]:
def input_ready(data_root: Path) -> bool:
    return (data_root / "raw_videos").exists() and (data_root / "metadata").exists()


if input_ready(AIC_DATA_ROOT):
    print(f"Đang dùng input có sẵn: {AIC_DATA_ROOT}")
elif AIC_ORGANIZER_SOURCE_URI:
    run_cli([
        "import-source",
        "--source-uri", AIC_ORGANIZER_SOURCE_URI,
        "--data-root", str(AIC_DATA_ROOT),
    ])
else:
    raise FileNotFoundError(
        "Chưa có input dataset. Hãy chuẩn bị layout:\n"
        f"{AIC_DATA_ROOT}/\n"
        "  raw_videos/\n"
        "  metadata/\n"
        "Hoặc set AIC_ORGANIZER_SOURCE_URI để notebook gọi system1 import-source."
    )

# Section 9 — Chạy ingest

`ingest` đọc raw video + metadata và tạo các output chính:

- `tables/videos.parquet`
- `raw_mapping/media_store_manifest.parquet`
- `manifests/dataset_report.json`

Nếu fail, kiểm tra lại input layout và metadata pairing.

In [ ]:
run_cli([
    "ingest",
    "--input", str(input_dir),
    "--output", str(output_dir),
])

# Section 10 — Chia batch

`assign-batches` tạo:

- `manifests/batch_manifest.csv`
- `manifests/batch_000.txt`
- các file batch tiếp theo nếu `AIC_NUM_BATCHES > 1`

`AIC_NUM_BATCHES` nên bằng số worker dự kiến. Notebook 01 và 02 phải dùng đúng `batch_id`.

In [ ]:
run_cli([
    "assign-batches",
    "--num-batches", str(num_batches),
    "--output", str(output_dir),
])

# Section 11 — Kiểm tra nhanh kết quả

User cần kiểm tra:

- số video có đúng không;
- `video_id` có đúng filename stem không;
- batch files có được tạo không;
- nếu dùng nhiều worker, phân batch có đúng số lượng không.

Report section xử lý thiếu file bằng cảnh báo tiếng Việt. Riêng `videos.parquet` thiếu thì báo lỗi rõ vì ingest chưa chạy hoặc đã fail.

In [ ]:
import csv

import pandas as pd
from IPython.display import JSON, display


def show_json_if_exists(path: Path, title: str) -> None:
    if path.exists():
        print(title)
        display(JSON(load_json(path)))
    else:
        print(f"Cảnh báo: chưa thấy {path}")


organizer_report_path = input_dir / "organizer_import_report.json"
show_json_if_exists(organizer_report_path, "organizer_import_report.json")

videos_path = release_dir / "tables" / "videos.parquet"
if not videos_path.exists():
    raise FileNotFoundError(f"Không thấy {videos_path}. Có thể ingest chưa chạy hoặc đã fail.")

videos = pd.read_parquet(videos_path)
print(f"videos_count={len(videos)}")
preferred_columns = [column for column in ["video_id", "video_ref"] if column in videos.columns]
if preferred_columns:
    display(videos[preferred_columns])
else:
    display(videos.head())

show_json_if_exists(release_dir / "manifests" / "dataset_report.json", "dataset_report.json")
show_json_if_exists(release_dir / "manifests" / "dataset_manifest.json", "dataset_manifest.json")

batch_manifest_path = release_dir / "manifests" / "batch_manifest.csv"
if batch_manifest_path.exists():
    with batch_manifest_path.open(encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    print(f"batch_manifest_rows={len(rows)}")
    display(pd.DataFrame(rows))
else:
    print(f"Cảnh báo: chưa thấy {batch_manifest_path}")

batch_paths = sorted((release_dir / "manifests").glob("batch_*.txt"))
if not batch_paths:
    print("Cảnh báo: chưa thấy file batch_*.txt")
for path in batch_paths:
    print(f"--- {path.name} ---")
    print(path.read_text(encoding="utf-8").strip())

# Section 12 — Gợi ý bước tiếp theo

Sau khi Notebook 00 pass, chạy tiếp:

```text
01_worker_structure_pipeline.ipynb
02_worker_feature_enrichment.ipynb
03_merge_validate_index_release.ipynb
```

Với debug test một worker:

- dùng `batch_id = "batch_000"`;
- dùng cùng `AIC_DATA_ROOT`;
- dùng cùng `AIC_RUNTIME_ROOT`.

Nếu chạy nhiều worker, mỗi worker dùng một `batch_id` khác nhau.

Nếu Kaggle session mất output, cần chạy lại hoặc restore checkpoint/output.